<a id="ksc-overview"></a>
# 02-2. PhysicsNeMo 신경 연산자 실습 — 2D Poisson FNO

**세션:** 16:10–17:30 (80분) · **선행:** [발사체 운동 PINN](01_Projectile_PINN.ipynb)

앞 실습의 PINN은 발사 조건 하나를 풀었습니다. 조건이 바뀌면 처음부터 다시 학습해야 했습니다.

이번에는 **함수에서 함수로 가는 규칙**을 학습시킵니다. 소스항 `f(x,y)`와 그에 대응하는 해 `u(x,y)`의 쌍을 2,000개쯤 보여 주면, 학습이 끝난 모델은 처음 보는 `f`를 받아 격자 전체의 `u`를 한 번에 내놓습니다. 이걸 신경 연산자(neural operator)라고 부르고, 그중 푸리에 공간에서 동작하는 것이 FNO입니다.

중간에 여러분이 직접 `-Δu=f`의 정확한 스펙트럴 해법을 한 줄로 짜 봅니다. 그 한 줄과 FNO가 각각 무엇을 알고 무엇을 모르는지 비교하는 게 이 실습의 핵심입니다.

## 오늘 이 노트북의 80분

| 절 | 내용 | 시간 |
|---:|---|---:|
| 1–3 | 환경 확인 · 문제 설정 · 실행 규모 선택 | 12분 |
| 4 | 데이터셋 준비와 검증 | 10분 |
| 5 | 입력장·출력장 관찰 | 6분 |
| 활동 1 | FFT 직접해법 한 줄 완성 | 8분 |
| 6–8 | 직접해법과 FNO의 차이 · FNO 내부 구조 · 코드 대응 | 14분 |
| 9–10 | 실험값 선택과 학습 실행 | 18분 |
| 11 | 결과 확인 | 4분 |
| 12 | GH200에서 HBM보다 큰 모델 다루기 | 4분 |
| 13 | 결과 해석 | 4분 |

10절 학습이 도는 동안 7–8절을 다시 읽어 두면 결과 해석이 빨라집니다.


## 1. 환경과 지원 파일 확인 — 3분

행사 환경에는 PhysicsNeMo 25.11, 데이터 생성기, 설정 파일이 SIF 이미지에 포함되어 있습니다. 아래 셀은 인터넷에 접속하지 않고 현재 파일과 GPU 상태만 확인합니다.


In [ ]:
from pathlib import Path
import json
import platform
import subprocess
import sys
import time

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "poisson_fno" / "generate_data.py").is_file()
        and (candidate / "labs" / "poisson_fno" / "train_fno.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("labs/poisson_fno를 찾지 못했습니다. 과정 폴더 안에서 노트북을 여세요.")

LAB_DIR = REPO_ROOT / "labs" / "poisson_fno"
required = [
    LAB_DIR / "data_validation.py",
    LAB_DIR / "generate_data.py",
    LAB_DIR / "train_fno.py",
    LAB_DIR / "notebook_utils.py",
    LAB_DIR / "images" / "fno_data_flow.svg",
    LAB_DIR / "conf" / "config_FNO.yaml",
    LAB_DIR / "conf" / "config_FNO_recovery.yaml",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("필수 파일이 없습니다: {}".format(missing))

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

import numpy as np
import torch
import physicsnemo
import physicsnemo.sym
from notebook_utils import ensure_profile_dataset, load_field_pair, profile_info

print("과정 루트       : {}".format(REPO_ROOT))
print("FNO 실습 폴더   : {}".format(LAB_DIR))
print("Python          : {}".format(sys.version.split()[0]))
print("시스템 아키텍처 : {}".format(platform.machine()))
print("PhysicsNeMo     : {}".format(getattr(physicsnemo, "__version__", "버전 정보 없음")))
print("CUDA 사용 가능  : {}".format(torch.cuda.is_available()))
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU             : {}".format(torch.cuda.get_device_name(0)))
    print("GPU 메모리      : {:.1f} GiB".format(properties.total_memory / 2**30))
else:
    print("경고: CUDA GPU를 사용할 수 없습니다. 이 셀의 전체 출력을 강사에게 전달하세요.")


## 2. 문제 설정 — 소스항이 달라질 때마다 달라지는 해 — 6분

단위 정사각형 `[0,1)²`에서 주기적 경계조건과 평균값이 0인 해를 사용합니다.

$$-\Delta u(x,y)=f(x,y)$$

- `f(x,y)`: 문제마다 달라지는 소스항 또는 강제항
- `u(x,y)`: 그 소스항에 대응하는 해
- `G:f→u`: 입력 함수 `f` 하나를 출력 함수 `u` 하나에 대응시키는 규칙

FNO는 하나의 고정된 해를 외우는 모델이 아닙니다. 여러 `(f,u)` 쌍에서 `G`를 학습한 뒤, 학습에 없던 새로운 `f`를 입력받아 그에 대응하는 `u`를 예측합니다.

푸리에 공간에서 0이 아닌 주파수 모드는 다음 관계를 만족합니다.

$$\hat u(k)=\frac{\hat f(k)}{(2\pi)^2|k|^2}, \qquad k\ne0$$

상수 모드는 `û(0)=0`으로 두어 평균값이 0인 해를 선택합니다.


## 3. 실행 설정과 표본 선택 — 3분

`gh200`은 256×256 격자와 큰 FNO를 사용하는 본 실습 설정입니다. `recovery`는 수업 진행을 복구할 때만 사용하는 단축 설정입니다. 두 설정은 데이터·모델·학습 단계가 함께 달라지므로 성능 비교용이 아닙니다.


In [ ]:
# [직접 수정 1] 기본 실습은 gh200을 사용합니다. 강사 안내가 있을 때만 recovery로 바꿉니다.
PROFILE = "gh200"

PROFILE_INFO = profile_info(LAB_DIR, PROFILE)
SAMPLE_INDEX = 0  # [직접 수정 2] 0 이상 테스트 표본 수 미만의 정수
test_count = int(PROFILE_INFO["split_sizes"]["test"])
if not 0 <= SAMPLE_INDEX < test_count:
    raise ValueError("SAMPLE_INDEX는 0 이상 {} 미만이어야 합니다.".format(test_count))

print("실행 설정       : {}".format(PROFILE))
print("격자            : {0} × {0}".format(PROFILE_INFO["grid_size"]))
print("데이터 수       : {}".format(PROFILE_INFO["split_sizes"]))
print("선택한 테스트 표본: {}".format(SAMPLE_INDEX))


## 4. 데이터셋 준비와 검증 — 10분

데이터 생성기는 주파수 범위를 제한한 임의의 평균 0 해 `u`를 먼저 만들고, 푸리에 공간에서 `f=-Δu`를 계산합니다. 반대 방향(먼저 `f`를 만들고 푸는 것)이 아닌 이유는, 이렇게 하면 정답 `u`가 수치 오차 없이 정확하기 때문입니다.

검증기는 방정식·실행 설정·배열 크기·Poisson 잔차를 확인합니다. 통과한 파일은 재사용하고, 없거나 조건이 다르면 세 분할을 다시 만듭니다.

> **처음 실행하면 시간이 걸립니다.** `gh200` 설정은 256×256 격자 표본을 2,560개(학습 2048 / 검증 256 / 테스트 256) 만들고 HDF5로 씁니다. 강사가 미리 생성해 둔 경우에는 검증만 하고 몇 초 만에 끝나지만, 새로 만드는 경우 수 분이 걸립니다. 아래 출력에서 `VALID`가 보이면 재사용, `REGENERATE`가 보이면 새로 생성 중입니다.
>
> 생성이 5분을 넘기거나 세션 시간이 빠듯하면 3절의 `PROFILE`을 `"recovery"`로 바꾸고 이 절부터 다시 실행합니다. 강사에게 먼저 확인하세요.


In [ ]:
FORCE_REGENERATE = False

_dataset_started = time.perf_counter()
PROFILE_INFO = ensure_profile_dataset(
    LAB_DIR,
    PROFILE,
    force=FORCE_REGENERATE,
)
DATASET_WALL_SECONDS = time.perf_counter() - _dataset_started
DATASET_DIR = PROFILE_INFO["dataset_dir"]
print("데이터셋 폴더: {}".format(DATASET_DIR))
print("이 단계 소요 시간: {:.2f} min".format(DATASET_WALL_SECONDS / 60.0))


## 5. 입력장과 출력장 관찰 — 6분

왼쪽 두 그림은 공간에서의 `f`와 `u`, 오른쪽 두 그림은 각 함수의 주파수 성분 크기입니다. 높은 주파수에서 `û(k)`는 `f̂(k)`를 `|k|²`에 비례하는 값으로 나눈 결과이므로, `u`가 일반적으로 더 부드럽습니다.


In [ ]:
import matplotlib.pyplot as plt

f_sample, u_sample = load_field_pair(
    LAB_DIR, PROFILE, split="test", sample_index=SAMPLE_INDEX
)

def log_spectrum(field):
    shifted = np.fft.fftshift(np.fft.fft2(field))
    return np.log10(1.0 + np.abs(shifted))

fields = (f_sample, u_sample, log_spectrum(f_sample), log_spectrum(u_sample))
titles = ("f (소스항)", "u (정답)", "log |FFT(f)|", "log |FFT(u)|")
figure, axes = plt.subplots(1, 4, figsize=(15, 3.6), constrained_layout=True)
for axis, field, title in zip(axes, fields, titles):
    image = axis.imshow(field, origin="lower", cmap="coolwarm")
    axis.set_title(title)
    axis.set_xlabel("격자 x")
    axis.set_ylabel("격자 y")
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
plt.show()

print("f 값의 범위: [{:.3e}, {:.3e}]".format(f_sample.min(), f_sample.max()))
print("u 값의 범위: [{:.3e}, {:.3e}]".format(u_sample.min(), u_sample.max()))


### 활동 1 — FFT 직접해법의 핵심 한 줄 완성 — 8분

아래 함수는 `f`를 푸리에 공간으로 바꾸고, 0이 아닌 모드에서 `û=f̂/((2π)²|k|²)`를 계산합니다. `u_hat[nonzero] = 0.0`의 오른쪽을 식에 맞게 바꾸십시오. 검사에 통과하면 직접 구한 해와 데이터셋의 정답이 일치합니다.


In [ ]:
def solve_poisson_fft(source):
    n = int(source.shape[0])
    if source.shape != (n, n):
        raise ValueError("source는 정사각형 2D 배열이어야 합니다.")

    source_hat = np.fft.rfft2(source)
    kx = np.fft.fftfreq(n, d=1.0 / n)
    ky = np.fft.rfftfreq(n, d=1.0 / n)
    kx_grid, ky_grid = np.meshgrid(kx, ky, indexing="ij")
    denominator = (2.0 * np.pi) ** 2 * (kx_grid**2 + ky_grid**2)
    nonzero = denominator > 0.0

    solution_hat = np.zeros_like(source_hat)
    # [직접 수정 3] 아래 0.0을 f_hat과 denominator를 사용한 식으로 바꿉니다.
    solution_hat[nonzero] = 0.0
    solution_hat[0, 0] = 0.0
    return np.fft.irfft2(solution_hat, s=source.shape).real


u_direct = solve_poisson_fft(f_sample)
direct_relative_l2 = np.linalg.norm(u_direct - u_sample) / np.linalg.norm(u_sample)
print("FFT 직접해와 데이터 정답의 상대 L2 오차: {:.6e}".format(direct_relative_l2))
if not direct_relative_l2 < 5.0e-4:
    raise AssertionError("직접해 코드가 아직 완성되지 않았습니다. 푸리에 공간 식을 다시 확인하세요.")


<details>
<summary><strong>정답 확인 · 푸리에 공간 계산</strong></summary>

```python
solution_hat[nonzero] = source_hat[nonzero] / denominator[nonzero]
```

0번 모드는 분모가 0이므로 나누지 않고 `0`으로 유지합니다. 이것이 평균값이 0인 해를 선택하는 조건입니다.
</details>


## 6. 직접해법과 FNO가 푸리에 공간을 사용하는 방식 — 3분

| 구분 | FFT 직접해법 | FNO |
|---|---|---|
| 푸리에 공간에서 하는 일 | 알려진 Poisson 식으로 각 모드를 나눔 | 데이터에서 선택한 모드의 가중치를 학습 |
| 필요한 지식 | 방정식의 정확한 스펙트럴 해법 | 여러 입력장·출력장 학습 예 |
| 결과 | 현재 `f` 한 개의 정확한 `u` | 보지 않은 `f`의 `u`를 빠르게 예측 |

직접해법 코드는 FNO의 정답을 만드는 간단한 기준선입니다. FNO가 정확한 Poisson 나눗셈을 그대로 실행하는 것은 아닙니다.


## 7. FNO 내부 구조 — 7분

<p align="center"><img src="../labs/poisson_fno/images/fno_data_flow.svg" width="1050" alt="소스항에서 예측 해까지의 FNO 데이터 흐름" /></p>

**모델 내부 흐름:** 입력 격자 `f`가 FNO 층을 지나 예측 격자 `u_pred`로 변환됩니다. `Domain`과 `Solver` 수준의 학습 구성은 다음 절에서 실제 코드와 대응합니다.

```text
f 격자 → 특징 채널 확장 → [FFT → 선택한 모드의 가중치 → 역 FFT
                              + 위치별 선형 변환 → 활성화] × L → 출력 변환 → u_pred 격자
```

- 특징 채널 확장(lifting): 한 채널의 `f`를 여러 내부 특징 채널로 바꿉니다.
- 주파수 경로: FFT 뒤 `fno_modes`만큼의 모드에 학습 가능한 가중치를 적용합니다.
- 위치별 경로: 각 격자 위치에서 선형 변환한 결과를 주파수 경로와 더합니다.
- 출력 변환(decoder): 내부 특징 채널을 한 채널의 `u_pred`로 바꿉니다.

데이터 생성기의 `max_mode`는 정답 데이터에 포함한 최고 주파수이고, 모델의 `fno_modes`는 각 FNO 층이 유지하는 푸리에 모드 수입니다.


## 8. PhysicsNeMo-Sym 구성과 코드 대응 — 4분

| 구성 요소 | `train_fno.py`의 코드 | 역할 |
|---|---|---|
| Hydra 설정 | `conf/config_FNO*.yaml` | 모델·배치·학습 단계 설정 |
| Dataset | `DictGridDataset(invar, outvar)` | HDF5의 `(f,u)` 쌍 제공 |
| Node | `fno.make_node("fno")` | `f → u_pred` 계산 그래프 |
| Constraint | `SupervisedGridConstraint` | 학습 데이터의 `u_pred`와 `u` 오차 최소화 |
| Validator | `GridValidator` | 가중치를 바꾸지 않고 검증 데이터 평가 |
| Domain | `domain.add_constraint`, `add_validator` | 학습·검증 구성 등록 |
| Solver | `Solver(cfg, domain).solve()` | 학습 반복과 체크포인트 저장 |
| Geometry / Inferencer / Monitor | 사용하지 않음 | 격자 지도학습 경로에는 등록하지 않음 |

별도 테스트 데이터는 무작위 초기 가중치의 기준 오차와 학습 종료 후 오차를 같은 분할에서 각각 한 번 측정합니다. 두 평가 모두 기울기를 계산하거나 가중치를 바꾸지 않습니다. 정규화의 평균과 표준편차는 학습 데이터에서만 계산합니다. 검증·테스트 데이터의 통계를 사용하면 평가할 정보가 학습 과정에 미리 들어가는 정보 누출이 생기기 때문입니다.


## 9. 모델과 학습 규모 선택 — 3분

설정 파일의 기본값을 읽은 뒤 `FNO_MODES`와 `MAX_STEPS`를 실험값으로 정합니다. `FNO_MODES`가 커지면 더 많은 주파수 성분을 직접 처리하지만 계산량과 모델 크기도 커질 수 있습니다.


In [ ]:
import yaml

config_path = LAB_DIR / "conf" / "{}.yaml".format(PROFILE_INFO["config_name"])
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

DEFAULT_FNO_MODES = int(config["arch"]["fno"]["fno_modes"])   # 설정 파일 기본값
DEFAULT_MAX_STEPS = int(config["training"]["max_steps"])      # 설정 파일 기본값

# [직접 수정 4] 두 값을 정합니다. 기본값을 그대로 쓰려면 아래 줄을 두고,
# 실험하려면 정수를 직접 씁니다.
#   FNO_MODES : 클수록 더 많은 주파수를 직접 처리하지만 모델과 계산량이 커집니다.
#               (gh200 기본 32 · 예: 16, 32, 48 / recovery 기본 12 · 예: 6, 12, 24)
#   MAX_STEPS : 학습 단계 수. 시간이 부족하면 줄입니다.
FNO_MODES = DEFAULT_FNO_MODES
MAX_STEPS = DEFAULT_MAX_STEPS

# 비교 실험을 할 때는 한 번에 한 값만 바꿉니다.

if not 1 <= FNO_MODES < int(PROFILE_INFO["grid_size"]) // 2:
    raise ValueError("FNO_MODES는 1 이상 Nyquist 모드 미만이어야 합니다.")
if MAX_STEPS < 1:
    raise ValueError("MAX_STEPS는 양의 정수여야 합니다.")

print("설정 파일       : {}".format(config_path.name))
print("데이터 max_mode : {} (데이터에 담긴 최고 주파수)".format(PROFILE_INFO["max_mode"]))
print("모델 fno_modes  : {} (기본값 {})".format(FNO_MODES, DEFAULT_FNO_MODES))
print("FNO 층 수       : {}".format(config["arch"]["fno"]["nr_fno_layers"]))
print("내부 채널 너비  : {}".format(config["arch"]["decoder"]["input_keys"][1]))
print("학습 배치 크기  : {}".format(config["batch_size"]["grid"]))
print("최대 학습 단계  : {} (기본값 {})".format(MAX_STEPS, DEFAULT_MAX_STEPS))


## 10. FNO 학습과 학습 전후 평가 — 15분

학습 전에는 무작위 초기 가중치로 테스트 오차를 한 번 측정합니다. 학습이 끝난 뒤 같은 테스트 분할을 다시 평가해 가중치 업데이트가 예측을 얼마나 바꾸었는지 자동 비교합니다. 테스트 데이터는 가중치 업데이트에 사용하지 않습니다.


기본 실행은 새 결과 폴더를 만듭니다. 중단된 실행을 이어갈 때만 강사가 확인한 기존 결과 폴더를 `RESUME_RUN_DIR`에 지정합니다. 데이터셋은 검증 후 재사용하지만 체크포인트와 평가 결과는 실행별로 분리됩니다.


In [ ]:
from datetime import datetime

RESUME_RUN_DIR = None  # 재개할 때만 기존 run 폴더의 절대 경로를 지정합니다.

if RESUME_RUN_DIR is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_OUTPUT_DIR = LAB_DIR / "outputs" / "ksc_fno_{}".format(PROFILE) / RUN_ID
    RUN_MODE = "새 학습"
else:
    RUN_OUTPUT_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    if not RUN_OUTPUT_DIR.is_dir():
        raise FileNotFoundError("재개할 run 폴더가 없습니다: {}".format(RUN_OUTPUT_DIR))
    RUN_MODE = "학습 재개"

METRICS_PATH = RUN_OUTPUT_DIR / "final_state_test_metrics.json"
train_command = list(PROFILE_INFO["train_command"])
train_command.extend(
    [
        "network_dir={}".format(RUN_OUTPUT_DIR),
        "custom.metrics_file={}".format(METRICS_PATH),
        "custom.test_example_index={}".format(SAMPLE_INDEX),
        "arch.fno.fno_modes={}".format(FNO_MODES),
        "training.max_steps={}".format(MAX_STEPS),
    ]
)
print("실행 방식 : {}".format(RUN_MODE))
print("결과 폴더 : {}".format(RUN_OUTPUT_DIR))
print("실행 명령 : {}".format(" ".join(str(part) for part in train_command)))

started = time.perf_counter()
completed = subprocess.run(train_command, cwd=LAB_DIR)
TRAIN_WALL_SECONDS = time.perf_counter() - started

print("학습 전체 시간: {:.2f} min".format(TRAIN_WALL_SECONDS / 60.0))
if completed.returncode != 0:
    raise RuntimeError("FNO 학습 실패 (exit code={})".format(completed.returncode))
if not METRICS_PATH.is_file():
    raise FileNotFoundError("평가 지표 파일을 찾지 못했습니다: {}".format(METRICS_PATH))


## 11. 학습 전후 테스트 오차와 예측 그림 — 4분

상대 L2 오차는 전체 테스트 분할에서 `||u_pred-u||₂/||u||₂`로 계산합니다. 학습 전후 오차의 비율과 별도 테스트 표본의 입력·정답·예측·절대 오차를 함께 확인합니다.


In [ ]:
from IPython.display import Image, display

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
assert metrics["profile"] == PROFILE
assert metrics["problem"] == "-Delta u = f"
assert metrics["model"]["fno_modes"] == FNO_MODES
assert metrics["max_steps_configured"] == MAX_STEPS

before = metrics["metrics_before_training"]
after = metrics["metrics_after_training"]
improvement = before["relative_l2"] / after["relative_l2"]

print("실행 결과 폴더      : {}".format(RUN_OUTPUT_DIR))
print("학습 전체 시간      : {:.2f} min".format(TRAIN_WALL_SECONDS / 60.0))
print("학습 전 상대 L2     : {:.6e}".format(before["relative_l2"]))
print("학습 후 상대 L2     : {:.6e}".format(after["relative_l2"]))
print("오차 감소 배수      : {:.2f}×".format(improvement))
print("학습 후 RMSE        : {:.6e}".format(after["rmse"]))
print("학습 후 MAE         : {:.6e}".format(after["mae"]))
print("학습 파라미터 수    : {:,}".format(metrics["model"]["trainable_parameters"]))

normalization = metrics["normalization_from_train_only"]
print("정규화 기준          : 학습 데이터만 사용")
print("  f: mean={:.3e}, std={:.3e}".format(normalization["f"]["mean"], normalization["f"]["std"]))
print("  u: mean={:.3e}, std={:.3e}".format(normalization["u"]["mean"], normalization["u"]["std"]))

peak_bytes = metrics["runtime_observation"]["peak_memory_allocated_bytes"]
if peak_bytes is not None:
    print("최대 PyTorch 메모리 : {:.2f} GiB".format(peak_bytes / 2**30))

test_figure = Path(metrics["artifacts"]["held_out_test_example"]["path"])
if not test_figure.is_file():
    raise FileNotFoundError(test_figure)
display(Image(filename=str(test_figure)))


In [ ]:
relative_l2_before = before["relative_l2"]
relative_l2_after = after["relative_l2"]
after_over_before = (
    relative_l2_after / relative_l2_before
    if relative_l2_before > 0.0
    else float("inf")
)

print("자동 학습 효과 요약")
print("학습 전 상대 L2 : {:.6e}".format(relative_l2_before))
print("학습 후 상대 L2 : {:.6e}".format(relative_l2_after))
print("학습 후/전 비율  : {:.4f}".format(after_over_before))
if after_over_before < 1.0:
    print("상대 L2 감소율   : {:.1f}%".format((1.0 - after_over_before) * 100.0))
elif after_over_before > 1.0:
    print("상대 L2 증가율   : {:.1f}%".format((after_over_before - 1.0) * 100.0))
else:
    print("상대 L2 변화     : 없음")

example_info = metrics["artifacts"]["held_out_test_example"]
print("표시한 테스트 표본: {}".format(example_info["sample_index"]))
print("표본 상대 L2     : {:.6e}".format(example_info["sample_relative_l2"]))
print("공간별 절대 오차는 위 그림의 |u_pred - u| 패널에서 확인합니다.")

# 이 실행의 숫자를 한 파일에 모아 둡니다.
RUN_SUMMARY_PATH = RUN_OUTPUT_DIR / "run_summary.json"
RUN_SUMMARY_PATH.write_text(
    json.dumps(
        {
            "lab": "poisson_fno",
            "profile": PROFILE,
            "grid_size": PROFILE_INFO["grid_size"],
            "split_sizes": PROFILE_INFO["split_sizes"],
            "data_max_mode": PROFILE_INFO["max_mode"],
            "fno_modes": FNO_MODES,
            "max_steps": MAX_STEPS,
            "sample_index": SAMPLE_INDEX,
            "dataset_wall_seconds": DATASET_WALL_SECONDS,
            "train_wall_seconds": TRAIN_WALL_SECONDS,
            "relative_l2_before": before["relative_l2"],
            "relative_l2_after": after["relative_l2"],
            "improvement_factor": improvement,
            "rmse_after": after["rmse"],
            "mae_after": after["mae"],
            "trainable_parameters": metrics["model"]["trainable_parameters"],
            "peak_memory_allocated_bytes": peak_bytes,
        },
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print("실행 요약 저장: {}".format(RUN_SUMMARY_PATH))


## 12. GH200에서 HBM보다 큰 모델 다루기 — 4분

방금 학습이 쓴 GPU 메모리를 이 GPU의 HBM 전체와 비교해 보세요. 한참 남았을 겁니다. 이 실습의 FNO는 작습니다.

그런데 실제 연구에서는 **모델이나 배치가 HBM에 안 들어가는 순간**이 옵니다. 보통 이렇게 끝납니다.

```
OutOfMemoryError: CUDA out of memory. Tried to allocate ...
```

일반적인 x86 + GPU 서버에서는 여기서 모델을 줄이거나 GPU를 더 붙여야 합니다. **GH200에는 선택지가 하나 더 있습니다.** PyTorch의 할당기를 `cudaMalloc` 대신 `cudaMallocManaged`로 바꾸면, HBM에 안 들어가는 부분이 NVLink-C2C를 통해 Grace의 LPDDR5X에 자리 잡습니다. 코드는 그대로 두고 할당기만 갈아 끼웁니다.

아래 셀은 같은 크기의 텐서를 **기본 할당기**와 **통합 메모리 할당기**로 각각 시도합니다. 할당기 교체는 CUDA를 처음 쓰기 전에 해야 하므로 두 조건을 별도 프로세스로 실행합니다.

> 성능이 공짜는 아닙니다. LPDDR5X는 HBM보다 느리므로 "일단 돌아가게 만드는" 수단이지 최적화가 아닙니다. 오전 GH200 실습에서 잰 C2C 대역폭이 이 방식의 상한을 정합니다.


In [ ]:
import re

UNIFIED_DEMO = LAB_DIR.parent / "gh200" / "pytorch_unified" / "oversubscribe_torch.py"
DEMO_BUILD_DIR = RUN_OUTPUT_DIR / "pytorch_unified_build"

if torch.cuda.is_available():
    total_hbm = torch.cuda.get_device_properties(0).total_memory
    print("이 학습이 쓴 최대 GPU 메모리: {:.2f} GiB".format(
        (peak_bytes or 0) / 2**30))
    print("이 GPU의 HBM 전체          : {:.2f} GiB".format(total_hbm / 2**30))
    if peak_bytes:
        print("사용 비율                  : {:.1f}%".format(
            100.0 * peak_bytes / total_hbm))

UNIFIED_RESULTS = {}
if not UNIFIED_DEMO.is_file():
    print("\n데모 스크립트를 찾지 못했습니다: {}".format(UNIFIED_DEMO))
else:
    for allocator in ("default", "managed"):
        print("\n" + "=" * 68)
        print("할당기: {}".format(allocator))
        completed = subprocess.run(
            [
                sys.executable, str(UNIFIED_DEMO),
                "--allocator", allocator,
                "--build-dir", str(DEMO_BUILD_DIR),
            ],
            capture_output=True,
            text=True,
        )
        match = re.search(r"^KSC_RESULT=(.*)$", completed.stdout, re.M)
        if match is None:
            print(completed.stdout[-2000:])
            print(completed.stderr[-2000:])
            print("결과를 해석할 수 없습니다. 위 출력을 강사에게 전달하세요.")
            continue
        payload = json.loads(match.group(1))
        UNIFIED_RESULTS[allocator] = payload
        print("요청 크기 : {:.2f} GiB (HBM 여유의 {:.2f}배)".format(
            payload["requested_bytes"] / 2**30,
            payload.get("requested_over_hbm_free") or float("nan"),
        ))
        print("결과      : {}".format(payload["status"]))
        if payload["status"] == "PASS":
            print("  할당 시간: {:.2f} s / 연산 시간: {:.2f} s / 값 검증: {}".format(
                payload["allocate_seconds"], payload["compute_seconds"],
                "OK" if payload.get("correct") else "확인 필요"))
        elif "detail" in payload:
            print("  {}".format(payload["detail"]))

if {"default", "managed"} <= set(UNIFIED_RESULTS):
    print("\n" + "=" * 68)
    print("{:12s} {:>10s}".format("할당기", "결과"))
    for name in ("default", "managed"):
        print("{:12s} {:>10s}".format(name, UNIFIED_RESULTS[name]["status"]))
    if (UNIFIED_RESULTS["default"]["status"] == "OOM"
            and UNIFIED_RESULTS["managed"]["status"] == "PASS"):
        print("\n기본 할당기로는 담지 못한 텐서를 통합 메모리 할당기로 처리했습니다.")
        print("HBM에 안 들어간 부분은 NVLink-C2C를 통해 Grace의 LPDDR5X에 있습니다.")

# 11절이 저장한 실행 요약에 이 절의 결과를 이어 붙입니다.
if UNIFIED_RESULTS and RUN_SUMMARY_PATH.is_file():
    summary = json.loads(RUN_SUMMARY_PATH.read_text(encoding="utf-8"))
    summary["unified_memory_demo"] = UNIFIED_RESULTS
    RUN_SUMMARY_PATH.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print("실행 요약 갱신: {}".format(RUN_SUMMARY_PATH))


## 13. 결과 해석과 실습 범위 — 4분

**이번 실습에서 확인한 것**

- FNO 하나가 여러 소스항에 대응하는 여러 해를 학습했습니다.
- 학습에 쓰지 않은 테스트 소스항에서 예측 오차를 쟀습니다.
- `fno_modes`, 학습 단계 수, 실행 시간, 모델 크기, GPU 메모리를 하나의 실험 기록으로 묶었습니다.

**강사와 함께 확인할 지점**

- FFT 직접해법은 Poisson 식을 **알고** 각 모드를 나눴습니다. FNO는 식을 모른 채 여러 입력장·출력장의 관계만 보고 배웠습니다. 두 방법이 각각 언제 유리한가.
- 학습 전후 상대 L2가 몇 배 줄었는가. 남은 오차는 그림의 어느 부분에 몰려 있는가.
- 정규화 통계를 학습 데이터에서만 계산한 이유는 무엇인가. 검증·테스트 통계를 쓰면 무엇이 잘못되는가.

## 참고 — 사전 검증에서 관측한 값

강사가 행사 전 KISTI PILOT GH200 한 대에서 `gh200` 기본 설정으로 측정한 값입니다. **정확히 같을 필요는 없습니다.**

| 항목 | 관측값 |
|---|---|
| 데이터셋 생성 시간 (최초 1회) | (행사 전 기입) 분 |
| 학습 2000 step 실행 시간 | (행사 전 기입) 분 |
| 학습 전 테스트 상대 L2 | (행사 전 기입) |
| 학습 후 테스트 상대 L2 | (행사 전 기입) |
| 오차 감소 배수 | (행사 전 기입) 배 |
| 학습 파라미터 수 | (행사 전 기입) |
| 최대 PyTorch GPU 메모리 | (행사 전 기입) GiB |

## 이 결과의 해석 범위

- 같은 격자 해상도의 테스트 분할만 평가했습니다. 다른 해상도로의 일반화나 수치해법 대비 추론 속도 우위는 이 실험이 보여 주지 않습니다.
- 난수 시드 하나, 한 번의 학습에서 얻은 관찰값입니다.
- `gh200`과 `recovery`는 격자·데이터·모델·학습 단계가 모두 다릅니다. 두 결과를 성능 비교로 읽지 않습니다.

필수 실습을 일찍 마쳤다면 강사 안내 후 [FNO 푸리에 모드 수 비교](optional/FNO_Mode_Ablation.ipynb)를 진행합니다.


---

## 참고 자료와 라이선스

공식 문서와 논문 링크는 [PhysicsNeMo 모듈 안내](README.md#참고-자료)에 모았습니다. 노트북 실행에는 인터넷 연결이 필요하지 않습니다.

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). Existing file-level notices remain in effect.
